# CineCompass: movie recommendation pipeline

This notebook is the full machine learning pipeline behind CineCompass: from raw data to the table your app reads. Each step explains **what it does and why**, so you can understand it, present it, and answer questions about it.

### The pipeline at a glance
```
1. Extract    download MovieLens 32M, verify it, load it
2. Check      first look, summary statistics, data quality tests
3. Explore    charts that explain the data
4. Transform  split the users, build the "liked" matrix, set up testing
5. Model      popularity, item-kNN and EASE, tuned on validation users
6. Evaluate   one final test on users the models never saw
7. Explain    a similarity heatmap and example picks with reasons
8. Enrich     sequels and directors from Wikidata, as simple rules
9. Load       CSV files -> MySQL -> PHP API -> React app
```

**There are 3 models.** Popularity is the baseline to beat. Item-kNN and EASE are the real recommenders.

### How to run it
Choose **Runtime > Run all**. It takes about 20 minutes on a normal CPU runtime; you don't need a GPU. At the end your browser downloads `item_neighbors.csv` and `items_seed.csv` (about 10 MB of plain data). Nothing else reaches your PC.

If Colab disconnects or runs out of memory: **Runtime > Restart session**, then **Run all** again. Files in Colab are deleted when the session ends.

### Words you'll see
| Word | Meaning |
|---|---|
| Liked matrix | A big table: one row per user, one column per movie, 1 if the user liked it |
| Sparse | Almost every cell is empty, so only the filled cells are stored |
| Cosine similarity | A 0-to-1 score for how often two movies are liked by the same people, adjusted so popular movies don't win just for being popular |
| Validation users | Users for choosing settings |
| Test users | Users kept aside until the very end, for the honest final score |
| Hit rate@10 | Share of users for whom at least one of the 10 picks was right |
| NDCG@10 | Like hit rate, but right picks near the top count more |
| Coverage | How many different movies the model ever recommends |
| Lambda | EASE's caution setting: higher means more careful weights |

---
# Stage 1. Extract

## 1.1 Download MovieLens 32M
This downloads the 239 MB dataset **into Colab, not onto your PC**. It prints two checksums. They must match: that proves the file is exactly what GroupLens published.

Licence: non-commercial use. Cite it as: F. Maxwell Harper and Joseph A. Konstan. 2015. *The MovieLens Datasets: History and Context.* ACM TiiS 5, 4.

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-32m.zip
!wget -q https://files.grouplens.org/datasets/movielens/ml-32m.zip.md5
!echo "Checksum of the downloaded file:" && md5sum ml-32m.zip
!echo "Checksum published by GroupLens:" && cat ml-32m.zip.md5
!unzip -q -o ml-32m.zip && ls -lh ml-32m

## 1.2 Imports and settings
These settings are design decisions. Each one is explained where it's used, and they belong in your report.

In [ ]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

SEED = 405          # makes the random user split repeatable
LIKE = 3.5          # 3.5 stars or more counts as "liked", like the app's thumbs up
RELEVANT = 4.0      # a hidden rating of 4 or more counts as a correct guess in testing
MIN_RATINGS = 200   # keep only movies with at least 200 ratings from training users
K = 50              # neighbours saved per movie, the rows of item_neighbors
TOP = 10            # picks shown to each user
N_VAL, N_TEST, N_TRAIN = 2500, 5000, 40000   # users in each group
LAMBDAS = [50, 200, 500, 1000, 2000, 5000]   # EASE settings to try
DATA = "ml-32m"
PURPLE = "#534AB7"
plt.rcParams["figure.figsize"] = (8, 4)

## 1.3 Load the files
- `ratings.csv`: one row per rating (user, movie, stars, time).
- `movies.csv`: title and genres for each movie.
- `links.csv`: each movie's IMDb ID, which connects MovieLens to OMDb and your MySQL tables.

In [ ]:
ratings = pd.read_csv(f"{DATA}/ratings.csv",
                      dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"})
movies = pd.read_csv(f"{DATA}/movies.csv")
links = pd.read_csv(f"{DATA}/links.csv")
title_of = dict(zip(movies.movieId, movies.title))
print(f"{len(ratings):,} ratings from {ratings.userId.nunique():,} users on {ratings.movieId.nunique():,} movies")
ratings.head()

---
# Stage 2. Check

Never trust data before looking at it. This stage goes from a quick look, to summary numbers, to automatic tests.

## 2.1 First look
For each file you'll see:
- **`head()`:** the first rows, so you know what one record looks like.
- **`info()`:** each column's type and the memory used.
- **Missing values:** `isna().sum()`, the number of empty cells per column.
- **Different values:** `nunique()`, for example how many different users and movies there are.

In [ ]:
for name, df in [("ratings", ratings), ("movies", movies), ("links", links)]:
    print(f"\n========== {name}: {len(df):,} rows ==========")
    display(df.head())
    df.info(memory_usage="deep")
    print("\nMissing values per column:")
    print(df.isna().sum().to_string())
    print("\nDifferent values per column:")
    print(df.nunique().to_string())

**What to notice:**
- **ratings** should have no missing values.
- **links** has some missing `tmdbId` values (about 124). That isn't a problem: you join on `imdbId`, which every movie has. A missing value only matters if you use that column.
- **Memory:** ratings uses compact types (`int32`, `float32`) set when loading. With the default types it would use about 60% more memory.
- **Timestamps** are seconds since 1 January 1970. The next cell turns them into dates.

## 2.2 Summary statistics
`describe()` gives the count, mean, spread and range of the star ratings.

In [ ]:
display(ratings[["rating"]].describe().T)
first, last = pd.to_datetime(ratings.timestamp.agg(["min", "max"]), unit="s")
print(f"First rating: {first:%Y-%m-%d}. Last rating: {last:%Y-%m-%d}")

**What to notice:** compare the mean with 2.75, the middle of the 0.5 to 5 scale. If it's clearly higher, people mostly rate movies they chose and enjoyed, so the middle of the scale isn't "neutral". That's why "liked" starts at 3.5 stars, not 2.75.

## 2.3 Data quality tests
Now the checks that matter for this project, as automatic tests. Every line should say PASS. If one says FAIL, stop and look before going on: a model trained on broken data gives confident, wrong answers.

In [ ]:
year_in_title = movies.title.str.extract(r"\((\d{4})\)\s*$")[0]
checks = {
    "every rating is between 0.5 and 5": ratings.rating.between(0.5, 5).all(),
    "every rating is a whole or half star": ((ratings.rating * 2) % 1 == 0).all(),
    "no missing values in ratings": not ratings.isna().any().any(),
    "no user rated the same movie twice": not ratings.duplicated(["userId", "movieId"]).any(),
    "every rated movie is in movies.csv": ratings.movieId.isin(movies.movieId).all(),
    "every movie has an IMDb ID": movies.movieId.isin(links.movieId).all() and links.imdbId.notna().all(),
}
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)
print(f"\nNot errors, just facts to know:")
print(f"  movies with no genre listed: {(movies.genres == '(no genres listed)').sum():,}")
print(f"  movies with no year in the title: {year_in_title.isna().sum():,}")

---
# Stage 3. Explore

Each chart answers a question, and each answer explains one of your design decisions. Read the printed numbers under each chart: they're your real numbers for the report.

## 3.1 How generous are people with stars?
**Why it matters:** you count 3.5 stars or more as "liked". This shows how much of the data that covers.

In [ ]:
share = ratings.rating.value_counts(normalize=True).sort_index()
share.plot(kind="bar", color=PURPLE, title="Share of ratings by stars")
plt.xlabel("Stars"); plt.ylabel("Share of all ratings"); plt.show()
print(f"Most common rating: {share.idxmax()} stars")
print(f"Ratings of {LIKE} stars or more (counted as liked): {share[share.index >= LIKE].sum():.1%}")

## 3.2 How many movies does each user rate?
**Why it matters:** MovieLens users rate a lot of movies, but your app's users start with zero. That gap is why the app mixes in popular movies for new users, and why stage 6 also tests with only each user's first 10 ratings.

In [ ]:
per_user = ratings.groupby("userId").size()
plt.hist(per_user, bins=np.logspace(np.log10(per_user.min()), np.log10(per_user.max()), 50), color=PURPLE)
plt.xscale("log"); plt.title("Ratings per user"); plt.xlabel("Ratings (log scale)"); plt.ylabel("Users"); plt.show()
print(f"Fewest ratings by one user: {per_user.min():,}. Median: {per_user.median():.0f}. Most: {per_user.max():,}")

## 3.3 Do a few movies get most of the attention?
**Why it matters:** this "long tail" is why popularity is a strong baseline, why you drop movies with few ratings (too little evidence to learn from), and why coverage is worth measuring: a lazy model just recommends the head of the tail.

In [ ]:
per_movie = ratings.groupby("movieId").size().sort_values(ascending=False)
plt.plot(np.arange(1, len(per_movie) + 1), per_movie.values, color=PURPLE)
plt.xscale("log"); plt.yscale("log"); plt.title("Ratings per movie, most rated first")
plt.xlabel("Movie rank (log scale)"); plt.ylabel("Ratings (log scale)"); plt.show()
top1 = per_movie.head(max(1, len(per_movie) // 100)).sum() / per_movie.sum()
print(f"{len(per_movie):,} movies have ratings. The top 1% of them get {top1:.0%} of all ratings.")
print(f"Movies with at least {MIN_RATINGS} ratings: {(per_movie >= MIN_RATINGS).sum():,}")

## 3.4 How empty is the user-movie table?
**Why it matters:** if every user had rated every movie, you wouldn't need a recommender. The table is almost entirely empty, and filling in the blanks is exactly the model's job. It's also why the code stores it as a sparse matrix.

In [ ]:
n_users, n_movies = ratings.userId.nunique(), ratings.movieId.nunique()
filled = len(ratings) / (n_users * n_movies)
print(f"{n_users:,} users x {n_movies:,} movies = {n_users * n_movies:,} possible ratings")
print(f"Filled: {filled:.3%}. Empty: {1 - filled:.3%}")

## 3.5 When are the ratings from, and what genres are the movies?
**Why it matters:** the data has an end date. Movies released after it can't be recommended, which is a limit to state in your report and in the app.

In [ ]:
years = pd.to_datetime(ratings.timestamp, unit="s").dt.year
years.value_counts().sort_index().plot(kind="bar", color=PURPLE, title="Ratings per year", figsize=(10, 4))
plt.xlabel("Year"); plt.ylabel("Ratings"); plt.show()
print(f"Ratings from {years.min()} to {years.max()}. Newest movie year in movies.csv: {pd.to_numeric(year_in_title).max():.0f}")
del years

movies.genres.str.split("|").explode().value_counts().plot(kind="barh", color=PURPLE, title="Movies per genre", figsize=(8, 6))
plt.gca().invert_yaxis(); plt.xlabel("Movies"); plt.show()

---
# Stage 4. Transform

## 4.1 Split the users
The models are tested on people they never saw, the same way a new person would use your app:
- **Validation users** choose settings, like EASE's lambda.
- **Test users** are used once, at the very end, for the score in your report.
- **Training users** are what the models learn from. 40,000 keeps it fast.

Splitting by *user* (not by rating) means no test user's taste leaks into training. That's called avoiding data leakage.

In [ ]:
rng = np.random.default_rng(SEED)
users = ratings.userId.unique()
rng.shuffle(users)
val_users = users[:N_VAL]
test_users = users[N_VAL:N_VAL + N_TEST]
train_users = users[N_VAL + N_TEST:N_VAL + N_TEST + N_TRAIN]
print(f"training {len(train_users):,}, validation {len(val_users):,}, test {len(test_users):,} users")

## 4.2 Build the "liked" matrix
`X` has one row per training user and one column per movie, with 1 where the user liked it. Two transformations happen here:
- **Filtering:** only movies with at least `MIN_RATINGS` training ratings. Fewer is too little evidence.
- **Binarising:** stars become liked (1) or not (blank). The app uses thumbs up, so the model learns the same signal.

In [ ]:
train = ratings[ratings.userId.isin(train_users)]
counts = train.groupby("movieId").size()
movie_ids = np.sort(counts[counts >= MIN_RATINGS].index.values)   # column number -> MovieLens movieId
movie_index = {m: i for i, m in enumerate(movie_ids)}               # MovieLens movieId -> column number
n = len(movie_ids)

likes = train[(train.rating >= LIKE) & train.movieId.isin(movie_ids)]
user_rows = likes.userId.astype("category").cat.codes.values
X = sp.csr_matrix((np.ones(len(likes), dtype=np.float32), (user_rows, likes.movieId.map(movie_index).values)),
                  shape=(user_rows.max() + 1, n))
popularity = np.asarray(X.sum(axis=0)).ravel()   # training likes per movie
print(f"{n:,} movies kept, {X.shape[0]:,} training users with likes, {X.nnz:,} likes")
print(f"Memory: sparse X uses about {X.data.nbytes + X.indices.nbytes + X.indptr.nbytes:,} bytes")

## 4.3 Set up the test
This is the testing protocol. Put it in your report.
1. Keep users with at least 10 ratings, sorted by time.
2. The first 80% are the **profile** (what the user told the app). The last 20% are **hidden**, like their future.
3. The model sees only the profile movies they liked.
4. A pick is **correct** if it's a hidden movie they rated 4 or more.

A second test list cuts each profile to the **first 10 ratings**, to see how models do for new users.

In [ ]:
def build_eval_users(user_ids, first_n=None):
    # For each user: (liked profile movies, all profile movies, set of relevant hidden movies)
    df = ratings[ratings.userId.isin(user_ids)].sort_values(["userId", "timestamp"], kind="stable")
    out = []
    for _, g in df.groupby("userId", sort=False):
        if len(g) < 10:
            continue
        cut = int(len(g) * 0.8)
        profile, hidden = g.iloc[:cut], g.iloc[cut:]
        if first_n:
            profile = profile.iloc[:first_n]
        profile = profile[profile.movieId.isin(movie_ids)]
        seen = profile.movieId.map(movie_index).values
        liked = profile.movieId[profile.rating >= LIKE].map(movie_index).values
        good = hidden[(hidden.rating >= RELEVANT) & hidden.movieId.isin(movie_ids)]
        relevant = set(good.movieId.map(movie_index).tolist()) - set(seen.tolist())
        if len(liked) and relevant:
            out.append((liked, seen, relevant))
    return out

val = build_eval_users(val_users)
test = build_eval_users(test_users)
test_first10 = build_eval_users(test_users, first_n=10)
print(f"validation {len(val):,}, test {len(test):,}, test with first 10 ratings {len(test_first10):,} users")

## 4.4 The scoring function
For each user it takes the top 10 picks (never a movie already in their profile) and computes hit rate, precision, recall, NDCG and coverage. Every model is judged by this same function, so the comparison is fair.

In [ ]:
DISCOUNT = 1 / np.log2(np.arange(2, TOP + 2))
TIE_BREAK = (1e-6 * popularity / popularity.max()).astype(np.float32)   # breaks exact ties by popularity

def evaluate(users, score_fn, batch=1000, decay=1.0):
    # decay < 1 makes older likes count less: the newest like counts 1, the one before it decay, then decay^2...
    hits, ndcgs, precisions, recalls, shown = [], [], [], [], set()
    for start in range(0, len(users), batch):
        chunk = users[start:start + batch]
        rows = np.concatenate([np.full(len(liked), j) for j, (liked, _, _) in enumerate(chunk)])
        cols = np.concatenate([liked for liked, _, _ in chunk])
        vals = np.concatenate([decay ** np.arange(len(liked) - 1, -1, -1) for liked, _, _ in chunk]).astype(np.float32)
        profiles = sp.csr_matrix((vals, (rows, cols)), shape=(len(chunk), n))
        scores = np.asarray(score_fn(profiles), dtype=np.float32) + TIE_BREAK
        for j, (_, seen, relevant) in enumerate(chunk):
            s = scores[j]
            s[seen] = -np.inf
            top = np.argpartition(-s, TOP)[:TOP]
            top = top[np.argsort(-s[top])]
            h = np.array([m in relevant for m in top], dtype=float)
            hits.append(h.any())
            precisions.append(h.sum() / TOP)
            recalls.append(h.sum() / len(relevant))
            ndcgs.append((h * DISCOUNT).sum() / DISCOUNT[:min(len(relevant), TOP)].sum())
            shown.update(top.tolist())
    summary = {"hit_rate": np.mean(hits), "precision": np.mean(precisions), "recall": np.mean(recalls),
               "ndcg": np.mean(ndcgs), "coverage": len(shown) / n}
    return summary, np.array(hits, dtype=float), np.array(ndcgs)

def show(name, summary):
    print(f"{name:<22}" + "  ".join(f"{k}={v:.3f}" for k, v in summary.items()))

---
# Stage 5. Model

## 5.1 How the models think: a tiny example
Before the real data, here's the whole idea on 6 people and 5 movies. 1 means they liked it.

In [ ]:
toy_movies = ["Godfather", "Goodfellas", "Casino", "Toy Story", "Finding Nemo"]
toy = pd.DataFrame([[1, 1, 1, 0, 0],
                    [1, 1, 0, 0, 0],
                    [1, 0, 1, 0, 0],
                    [0, 0, 0, 1, 1],
                    [0, 1, 0, 1, 1],
                    [0, 0, 0, 0, 1]],
                   index=["Ali", "Sara", "Omar", "Lina", "Noura", "Fahad"], columns=toy_movies)
toy

**Step 1: count co-likes.** `G = X.T @ X`. Each cell says how many people liked *both* movies. The diagonal is how many liked each movie.

In [ ]:
G_toy = toy.T @ toy
G_toy

**Step 2: item-kNN similarity.** Divide each count by the square root of both movies' totals (cosine similarity). Now a movie liked by everyone doesn't look similar to everything.

In [ ]:
G_arr = G_toy.values.astype(float)
d_toy = np.sqrt(np.diag(G_arr))
cos_arr = G_arr / np.outer(d_toy, d_toy)
np.fill_diagonal(cos_arr, 0)
cos_toy = pd.DataFrame(cos_arr, index=toy_movies, columns=toy_movies)
cos_toy.round(2)

**Step 3: recommend.** A new user likes only *Godfather*. Their scores are simply the *Godfather* row: the higher the number, the stronger the pick, and the reason is "because you liked *Godfather*".

In [ ]:
print("Item-kNN scores for a fan of Godfather:")
print(cos_toy.loc["Godfather"].drop("Godfather").sort_values(ascending=False).round(2))

**Step 4: EASE does the same job differently.** Instead of a fixed formula, it *learns* the weights that best rebuild everyone's likes from their other likes, with lambda stopping it from over-trusting tiny counts. Same kind of table, same way of recommending.

Notice that EASE can give **negative** weights: liking *Godfather* makes *Toy Story* a slightly worse guess. Item-kNN can't express that, which is one reason EASE scored higher in testing.

In [ ]:
P_toy = np.linalg.inv(G_arr + 1.0 * np.eye(len(toy_movies)))   # lambda = 1 for this tiny example
B_arr = P_toy / (-np.diag(P_toy))
np.fill_diagonal(B_arr, 0)
B_toy = pd.DataFrame(B_arr, index=toy_movies, columns=toy_movies)
print("EASE scores for a fan of Godfather:")
print(B_toy.loc["Godfather"].drop("Godfather").sort_values(ascending=False).round(2))

## 5.2 Model 1: popularity baseline
Everyone gets the same list: the most-liked movies they haven't rated. It has no idea who you are, so any real model must beat it. **Check:** hit rate roughly 0.3.

In [ ]:
pop_scores = (popularity / popularity.max()).astype(np.float32)

def popularity_model(profiles):
    return np.tile(pop_scores, (profiles.shape[0], 1))

show("Popularity", evaluate(val, popularity_model)[0])

## 5.3 Model 2: item-kNN
Steps 1 to 3 of the tiny example, on the real data. Each movie keeps its 50 most similar movies: that's the `item_neighbors` table. **Check:** hit rate roughly 0.45.

In [ ]:
def top_k_per_row(M, k):
    # Keep the k largest positive values in each row of a dense matrix, as a sparse matrix
    idx = np.argpartition(-M, k, axis=1)[:, :k]
    vals = np.maximum(np.take_along_axis(M, idx, axis=1), 0)
    rows = np.repeat(np.arange(M.shape[0]), k)
    S = sp.csr_matrix((vals.ravel().astype(np.float32), (rows, idx.ravel())), shape=M.shape)
    S.eliminate_zeros()
    return S

t = time.time()
G = (X.T @ X).toarray().astype(np.float64)
norms = np.sqrt(np.diag(G)) + 1e-9
C = G / np.outer(norms, norms)
np.fill_diagonal(C, 0)
S_knn = top_k_per_row(C, K)
del C
print(f"item-kNN built in {time.time() - t:.1f}s")

def knn_model(profiles):
    return (profiles @ S_knn).toarray()

show("Item-kNN", evaluate(val, knn_model)[0])

## 5.4 Model 3: EASE
Step 4 of the tiny example, on the real data:
1. `P = inverse(G + lambda * I)`
2. `B[i, j] = -P[i, j] / P[j, j]`, with `B[i, i] = 0`

Lambda is **tuned**: each value is scored on validation users, and the best one wins. This is the only "training choice" in the model, and making it on validation users (never test users) keeps the final score honest. Then each movie keeps its 50 strongest positive weights (`B50`), because that's what fits in MySQL.

**Check:** hit rate roughly 0.55 for full EASE, slightly lower for `B50`.

Cite: Harald Steck. 2019. *Embarrassingly Shallow Autoencoders for Sparse Data.* The Web Conference (WWW '19).

In [ ]:
def train_ease(lam):
    A = G.copy()
    A[np.diag_indices(n)] += lam
    P = np.linalg.inv(A)
    del A
    B = P / (-np.diag(P))
    np.fill_diagonal(B, 0)
    return B.astype(np.float32)

tuning = []
for lam in LAMBDAS:
    B = train_ease(lam)
    summary = evaluate(val, lambda p: p @ B)[0]
    tuning.append({"lambda": lam, **summary})
    show(f"EASE lambda={lam}", summary)
tuning = pd.DataFrame(tuning)
best_lambda = int(tuning.loc[tuning.ndcg.idxmax(), "lambda"])
print(f"best lambda on validation users: {best_lambda}")

tuning.plot(x="lambda", y="ndcg", marker="o", logx=True, color=PURPLE, legend=False,
            title="EASE tuning on validation users")
plt.ylabel("NDCG@10"); plt.show()

B = train_ease(best_lambda)
B50 = top_k_per_row(B, K)

def ease_full_model(profiles):
    return profiles @ B

def ease_model(profiles):
    return (profiles @ B50).toarray()

show("EASE, top 50", evaluate(val, ease_model)[0])

## 5.5 Do recent likes matter more?
Taste changes. Someone who just liked five Hitchcock films probably wants Hitchcock *now*, even if most of their older likes are modern films. So each older like can count a little less:
- **decay 0.9:** the newest like counts 1, the one before 0.9, then 0.81, and so on.
- **decay 1.0:** every like counts the same (no recency).

Like lambda, the best decay is chosen on **validation** users. If 1.0 wins, recency doesn't help on this data, and you keep it off. The profile in step 4.3 is already in time order, so this test is fair: the hidden movies really are each user's *next* likes.

In [ ]:
DECAYS = [1.0, 0.98, 0.95, 0.9, 0.8]
decay_tuning = []
for dcy in DECAYS:
    summary = evaluate(val, ease_model, decay=dcy)[0]
    decay_tuning.append({"decay": dcy, **summary})
    show(f"EASE, decay={dcy}", summary)
decay_tuning = pd.DataFrame(decay_tuning)
best_decay = float(decay_tuning.loc[decay_tuning.ndcg.idxmax(), "decay"])
print(f"best decay on validation users: {best_decay}"
      + ("  (recency doesn't help here, so every like counts the same)" if best_decay == 1.0 else ""))

---
# Stage 6. Evaluate (run once)

The test users, used for the first and only time. Don't change any setting after seeing these numbers: that would quietly tune on the test and make the score dishonest. This table goes in your report.

For comparison, an earlier exploratory run got hit rates of about 0.32 (popularity), 0.46 (item-kNN) and 0.54 (EASE, top 50) with full profiles. Yours will differ a little because your split is different.

In [ ]:
models = {"Popularity": (popularity_model, 1.0), "Item-kNN": (knn_model, 1.0),
          "EASE, top 50": (ease_model, 1.0), "EASE, full": (ease_full_model, 1.0),
          "EASE, top 50, recent likes": (ease_model, best_decay)}
rows, per_user = [], {}
for profile_name, users_list in [("Full profile", test), ("First 10 ratings", test_first10)]:
    for name, (fn, dcy) in models.items():
        summary, hits, ndcgs = evaluate(users_list, fn, decay=dcy)
        rows.append({"profile": profile_name, "model": name, **summary})
        per_user[profile_name, name] = hits
final = pd.DataFrame(rows).round(3)

final.pivot(index="model", columns="profile", values="hit_rate").loc[list(models)].plot(
    kind="bar", color=[PURPLE, "#AFA9EC"], title="Hit rate@10 on test users", rot=0)
plt.ylabel("Hit rate@10"); plt.show()
final

**Is the difference real or luck?** This resamples the test users 2,000 times. If the 95% interval doesn't include 0, the difference is real.

In [ ]:
def bootstrap_diff(a, b, reps=2000):
    d = a - b
    means = d[np.random.default_rng(SEED).integers(0, len(d), (reps, len(d)))].mean(axis=1)
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

for profile_name in ["Full profile", "First 10 ratings"]:
    for a, b in [("EASE, top 50", "Item-kNN"), ("Item-kNN", "Popularity"), ("EASE, top 50, recent likes", "EASE, top 50")]:
        m, lo, hi = bootstrap_diff(per_user[profile_name, a], per_user[profile_name, b])
        print(f"{profile_name}: {a} minus {b}, hit rate {m:+.3f} (95% interval {lo:+.3f} to {hi:+.3f})")

---
# Stage 7. Explain

## 7.1 What did the model learn? A similarity heatmap
Each square is how similar two movies are (item-kNN cosine similarity). Darker means "liked by the same people". Look for dark blocks: crime classics together, *Star Wars* together, animation together. Nobody told the model about genres; it found these groups from likes alone. This is a great slide for your video.

In [ ]:
showcase = ["Godfather, The (1972)", "Godfather: Part II, The (1974)", "Goodfellas (1990)", "Casino (1995)",
            "Star Wars: Episode IV - A New Hope (1977)", "Star Wars: Episode V - The Empire Strikes Back (1980)",
            "Star Wars: Episode VI - Return of the Jedi (1983)", "Toy Story (1995)", "Toy Story 2 (1999)",
            "Finding Nemo (2003)", "Notebook, The (2004)", "Titanic (1997)", "Pride & Prejudice (2005)",
            "Inception (2010)", "Interstellar (2014)", "Dark Knight, The (2008)"]
index_of_title = {t: movie_index[m] for m, t in title_of.items() if m in movie_index}
found = [t for t in showcase if t in index_of_title]
if len(found) < len(showcase):
    print("Not in the model, skipped:", [t for t in showcase if t not in index_of_title])
idx = [index_of_title[t] for t in found]
sub = G[np.ix_(idx, idx)]
d_sub = np.sqrt(np.diag(sub))
sim = sub / np.outer(d_sub, d_sub)
np.fill_diagonal(sim, np.nan)
labels = [t.rsplit(" (", 1)[0] for t in found]
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim, cmap="Purples")
ax.set_xticks(range(len(found)))
ax.set_xticklabels(labels, rotation=90)
ax.set_yticks(range(len(found)))
ax.set_yticklabels(labels)
fig.colorbar(im, label="Cosine similarity")
ax.set_title("How similar the model thinks movies are")
plt.tight_layout(); plt.show()

## 7.2 Example picks with reasons
A pretend user who liked *Casino*, *The Godfather* and *Goodfellas*. Each pick shows its reason, exactly like the app will.

In [ ]:
def recommend_for_titles(titles, model_matrix, top=TOP):
    liked = [movie_index[m] for m, t in title_of.items() if t in titles and m in movie_index]
    if not liked:
        print("  none of these titles are in the model")
        return
    profile = sp.csr_matrix((np.ones(len(liked), dtype=np.float32), (np.zeros(len(liked), dtype=int), liked)),
                            shape=(1, n))
    scores = (profile @ model_matrix).toarray()[0]
    scores[liked] = -np.inf
    weights = model_matrix[liked].toarray()
    for i in np.argsort(-scores)[:top]:
        because = title_of[movie_ids[liked[int(np.argmax(weights[:, i]))]]]
        print(f"  {title_of[movie_ids[i]]}  (because you liked {because})")

mafia = {"Casino (1995)", "Godfather, The (1972)", "Goodfellas (1990)"}
print("Item-kNN:")
recommend_for_titles(mafia, S_knn)
print("EASE:")
recommend_for_titles(mafia, B50)

---
# Stage 8. Enrich: sequels and directors

The model learns taste from ratings, so it doesn't *know* that *The Dark Knight* follows *Batman Begins*, or that Christopher Nolan made both. It often finds them anyway, because fans watch both, but not always and not in order. This stage adds those facts as simple **rules**, kept separate from the model:
- **`next_in_series`:** the films that come after this one in its series, in order. Score 1 for the next film, 0.5 for the one after, and so on.
- **`same_director`:** other films by the same director, best-known first.

**Where the facts come from:** [Wikidata](https://www.wikidata.org), the free database behind Wikipedia. Its data is CC0, meaning free for any use. The notebook sends it the IMDb IDs of your movies and gets back plain text answers (JSON) that stay in Colab's memory. Nothing runs from them, and nothing touches your PC. It takes about 1 to 2 minutes.

**Limits:** Wikidata is written by volunteers, so a few links are missing. If Wikidata doesn't answer, this stage prints a warning and the notebook carries on without these rows.

In [ ]:
import json, urllib.parse, urllib.request

imdb_of = {m: "tt" + str(i).zfill(7) for m, i in zip(links.movieId, links.imdbId)}
model_imdb = [imdb_of[m] for m in movie_ids]
in_model = set(model_imdb)
name_of_imdb = {imdb_of[m]: title_of[m] for m in movie_ids}
year_of = dict(zip(movies.movieId.map(imdb_of), pd.to_numeric(year_in_title)))
pop_of = {imdb_of[m]: float(popularity[movie_index[m]]) for m in movie_ids}

QUERY = '''SELECT ?imdb ?seriesName ?nextImdb ?prevImdb ?director ?directorName WHERE {
  VALUES ?imdb { %s }
  ?film wdt:P345 ?imdb .
  OPTIONAL { ?film wdt:P179 ?series .
    OPTIONAL { ?series rdfs:label ?sEn . FILTER(LANG(?sEn) = "en") }
    OPTIONAL { ?series rdfs:label ?sMul . FILTER(LANG(?sMul) = "mul") }
    BIND(COALESCE(?sEn, ?sMul) AS ?seriesName) }
  OPTIONAL { ?film wdt:P156 ?n . ?n wdt:P345 ?nextImdb . }
  OPTIONAL { ?film wdt:P155 ?p . ?p wdt:P345 ?prevImdb . }
  OPTIONAL { ?film wdt:P57 ?director .
    OPTIONAL { ?director rdfs:label ?dEn . FILTER(LANG(?dEn) = "en") }
    OPTIONAL { ?director rdfs:label ?dMul . FILTER(LANG(?dMul) = "mul") }
    BIND(COALESCE(?dEn, ?dMul) AS ?directorName) }
}'''
COLUMNS = ["imdb", "seriesName", "nextImdb", "prevImdb", "director", "directorName"]

def ask_wikidata(ids, batch=400):
    # P345 = IMDb ID, P179 = part of the series, P156 = followed by, P155 = follows, P57 = director
    # Names are asked for in English ("en"), falling back to Wikidata's language-neutral names ("mul")
    rows = []
    for start in range(0, len(ids), batch):
        values = " ".join(f'"{i}"' for i in ids[start:start + batch])
        body = urllib.parse.urlencode({"query": QUERY % values}).encode()
        req = urllib.request.Request("https://query.wikidata.org/sparql", data=body, headers={
            "Accept": "application/sparql-results+json",
            "User-Agent": "CineCompass/1.0 (university student project)"})
        for attempt in range(4):
            try:
                with urllib.request.urlopen(req, timeout=90) as r:
                    rows += [{k: v["value"] for k, v in b.items()} for b in json.load(r)["results"]["bindings"]]
                break
            except Exception as e:
                print(f"  batch {start}: {e}; retrying")
                time.sleep(5 * (attempt + 1))
        else:
            raise RuntimeError("Wikidata did not answer")
        time.sleep(1)
    return pd.DataFrame(rows, columns=COLUMNS)

t = time.time()
try:
    wd = ask_wikidata(model_imdb)
    print(f"Wikidata answered in {time.time() - t:.0f}s")
except RuntimeError as e:
    wd = pd.DataFrame(columns=COLUMNS)
    print("Wikidata unavailable, so there are no series or director rows this time:", e)

director_of = wd.dropna(subset=["directorName"]).groupby("imdb").directorName.agg(lambda s: "|".join(sorted(set(s))))
named_series = wd.dropna(subset=["seriesName"])
named_series = named_series[~named_series.seriesName.str.lower().str.startswith("list of")]   # e.g. "list of Pixar films"
series_of = named_series.groupby("imdb").seriesName.agg(lambda s: "|".join(sorted(set(s))))
print(f"Movies found: {wd.imdb.nunique():,} of {len(model_imdb):,}. "
      f"With a director: {len(director_of):,}. In a series: {len(series_of):,}")

**Build the series chains.** "Followed by" and "follows" links are combined (so one missing link can be filled by the other), kept only when both films are in the model and the order of years makes sense, then followed up to 5 films ahead.

In [ ]:
pairs = set(wd.dropna(subset=["nextImdb"])[["imdb", "nextImdb"]].itertuples(index=False, name=None))
pairs |= {(b, a) for a, b in wd.dropna(subset=["prevImdb"])[["imdb", "prevImdb"]].itertuples(index=False, name=None)}
pairs = {(a, b) for a, b in pairs
         if a in in_model and b in in_model and a != b and not year_of.get(b, 9999) < year_of.get(a, 0)}
next_of = {}
for a, b in pairs:
    next_of.setdefault(a, set()).add(b)

series_rows = []
for first in next_of:
    frontier, seen = {first}, {first}
    for depth in range(1, 6):
        frontier = {b for a in frontier for b in next_of.get(a, ()) if b not in seen}
        if not frontier:
            break
        seen |= frontier
        series_rows += [(first, b, round(1 / depth, 3), "next_in_series") for b in frontier]
series_rows = pd.DataFrame(series_rows, columns=["imdb_id", "neighbor_imdb_id", "score", "method"])

films_by_director = wd.dropna(subset=["director"]).groupby("director").imdb.agg(lambda s: sorted(set(s) & in_model))
top_pop = max(pop_of.values())
director_rows = pd.DataFrame(
    [(a, b, round(pop_of[b] / top_pop, 4), "same_director")
     for films in films_by_director if len(films) > 1 for a in films for b in films if a != b],
    columns=["imdb_id", "neighbor_imdb_id", "score", "method"]).drop_duplicates(["imdb_id", "neighbor_imdb_id"])
print(f"next_in_series rows: {len(series_rows):,}. same_director rows: {len(director_rows):,}")

imdb_by_title = {title_of[m]: imdb_of[m] for m in movie_ids}
rules = pd.concat([series_rows, director_rows])
for t, method in [("Batman Begins (2005)", "next_in_series"), ("Hangover, The (2009)", "next_in_series"),
                  ("Toy Story (1995)", "next_in_series"), ("Interstellar (2014)", "same_director")]:
    i = imdb_by_title.get(t)
    r = rules[(rules.imdb_id == i) & (rules.method == method)].sort_values("score", ascending=False).head(6)
    print(f"  {t} -> {method}: " + (", ".join(name_of_imdb[b] for b in r.neighbor_imdb_id) or "none"))

---
# Stage 9. Load

## 9.1 Export the CSV files
- **`item_neighbors.csv`:** `imdb_id, neighbor_imdb_id, score, method`. The winning model's rows (`ease`), plus the rule rows from stage 8 (`next_in_series` and `same_director`).
- **`items_seed.csv`:** `imdb_id, title, year, genres, director, series, popularity` for every movie in the model. Titles are cleaned: the year is removed and "Godfather, The" becomes "The Godfather". Names with more than one value are separated by `|`, like genres.
- **`popularity`** is how many training users liked the movie. The app uses it for new users: the onboarding grid shows well-known movies first, and the "Popular movies" row sorts by it.

If item-kNN won in stage 6, change the `WINNER` line to `S_knn, "knn"`.

**Check:** about 221,000 `ease` rows plus the rule rows, and a file of roughly 10 MB.

In [ ]:
WINNER, WINNER_NAME = B50, "ease"

imdb_of = {m: "tt" + str(i).zfill(7) for m, i in zip(links.movieId, links.imdbId)}
coo = WINNER.tocoo()
model_rows = pd.DataFrame({
    "imdb_id": [imdb_of[movie_ids[i]] for i in coo.row],
    "neighbor_imdb_id": [imdb_of[movie_ids[j]] for j in coo.col],
    "score": np.round(coo.data, 5),
    "method": WINNER_NAME,
})
neighbors = pd.concat([model_rows, series_rows, director_rows], ignore_index=True)
neighbors.to_csv("item_neighbors.csv", index=False)

items = pd.DataFrame({"movieId": movie_ids}).merge(movies, on="movieId")
items["imdb_id"] = items.movieId.map(imdb_of)
items["year"] = items.title.str.extract(r"\((\d{4})\)\s*$")[0]
items["title"] = (items.title.str.replace(r"\s*\(\d{4}\)\s*$", "", regex=True)
                  .str.replace(r"^(.*), (The|A|An)$", r"\2 \1", regex=True))
items["director"] = items.imdb_id.map(director_of).fillna("")
items["series"] = items.imdb_id.map(series_of).fillna("")
items["popularity"] = items.movieId.map(lambda m: int(popularity[movie_index[m]]))
items[["imdb_id", "title", "year", "genres", "director", "series", "popularity"]].to_csv("items_seed.csv", index=False)
print("Most popular:", ", ".join(items.sort_values("popularity", ascending=False).title.head(5)))
print(f"{len(items):,} movies. Rows per method:")
print(neighbors.method.value_counts().to_string())
print(f"\nSetting for your PHP API and tools/try_recommender.py: RECENT_DECAY = {best_decay}")

try:
    from google.colab import files
    files.download("item_neighbors.csv")
    files.download("items_seed.csv")
except ImportError:
    print("Not running in Colab, so the files were saved next to this notebook.")

## 9.2 Import into MySQL
The last step happens outside Colab. Either use phpMyAdmin's **Import** tab, or SQL like the example below. It's an example: adjust the table and column names to your final schema. It loads the movies, then swaps IMDb IDs for your own `item_id` numbers using a temporary table.

```sql
LOAD DATA LOCAL INFILE 'items_seed.csv' INTO TABLE items
  FIELDS TERMINATED BY ',' OPTIONALLY ENCLOSED BY '"' IGNORE 1 LINES
  (imdb_id, title, @year, genres, director, series, popularity)
  SET kind = 'movie', year = NULLIF(@year, '');

CREATE TEMPORARY TABLE neighbors_staging (
  imdb_id VARCHAR(12), neighbor_imdb_id VARCHAR(12), score FLOAT, method VARCHAR(20));
LOAD DATA LOCAL INFILE 'item_neighbors.csv' INTO TABLE neighbors_staging
  FIELDS TERMINATED BY ',' IGNORE 1 LINES;

INSERT INTO item_neighbors (item_id, neighbor_id, score, method)
SELECT a.item_id, b.item_id, s.score, s.method
FROM neighbors_staging s
JOIN items a ON a.imdb_id = s.imdb_id
JOIN items b ON b.imdb_id = s.neighbor_imdb_id;
```

From here, PHP reads `item_neighbors` by `method`, and React shows the rows:
- **`ease`:** "Picked for you" and "More like *Casino*".
- **`next_in_series`:** "Continue the series", in order of `score` (1 = the next film).
- **`same_director`:** "More from Christopher Nolan", with the name taken from `items.director`.

---
# Interview questions you should be ready for

Fill in the X values from your own run.

**How did you check the data?**
I looked at every file first (rows, types, missing values, unique counts), then ran automatic tests: valid star values, no duplicate ratings, every movie has an IMDb ID. Some `tmdbId` values were missing, but that didn't matter because I joined on IMDb IDs. *(Stage 2)*

**Why didn't you predict star ratings and measure error (RMSE)?**
The app shows a top-10 list, so I measured what users actually see: whether the list contains movies they went on to like (hit rate, NDCG). A model can predict stars well and still rank badly. *(4.4)*

**How did you avoid data leakage?**
I split by user, so no test user's taste was in training. Each test user's ratings were split by time, so the model never saw their future. I tuned lambda on validation users and used the test users once. *(4.1, 4.3, stage 6)*

**Why EASE?**
I compared three models on the same test users. EASE beat item-kNN by X points of hit rate, and the bootstrap interval showed it wasn't luck. It also produces the same kind of table, so the app didn't change. *(Stage 6)*

**What about new users with few ratings (cold start)?**
I tested with only the first 10 ratings: every model was close to the popularity baseline. So the app mixes in popular movies and shows "More like X" rows until a user has about 20 likes. *(Stage 6)*

**How does it explain a pick?**
A pick's score is a sum of contributions from the movies the user liked. The biggest contributor is the reason: "Because you liked *Casino*". *(7.2)*

**Why not deep learning?**
The data is a large, sparse liked matrix. EASE is a simple linear model that trains in seconds and is easy to explain, and published comparisons show models like it are competitive with deep models on this kind of data. *(5.4)*

**What if a user's taste changes?**
Older likes count less: each step back in time multiplies a like's weight by a decay value. I chose the decay on validation users, and the final test compares it against equal weights, so it's only used if it helps. *(5.5, stage 6)*

**Is everything machine learning?**
No, on purpose. The model learns taste from ratings. Facts such as "*The Dark Knight* follows *Batman Begins*" come from Wikidata as rules, because a fact shouldn't depend on how many people happened to rate both films. Combining the two is called a hybrid recommender. *(Stage 8)*

**What are the limits, and what would you do next?**
MovieLens users aren't Saudi 18-30 year olds, the data ends in 2023, and there's no TV. Next: retrain on CineCompass's own likes once users join, and add a TV model based on show descriptions.

---
# Report, CV and AI citation

**Put in your report:** the pipeline diagram at the top, the data checks (stage 2), two or three charts from stage 3 with what they told you, the testing protocol (4.3), the results table and intervals (stage 6), the heatmap (7.1), and the limits: MovieLens users aren't Saudi 18-30 year olds, the data ends in 2023, and every model is weak for users with only a few ratings.

**A CV line, once you have your numbers:**
> Built an end-to-end recommendation pipeline (ETL, data validation, EDA, model tuning, held-out evaluation) on 32M MovieLens ratings. Compared popularity, item-kNN and EASE; EASE improved hit rate@10 by X points over item-kNN. Served results through a PHP/MySQL API to a React app.

**AI citation:** cite the tool, the date and the exact prompt. For example:
> Claude Opus 5.5 (Anthropic), 09/11/2026: "paste your exact prompt here". Used for: this notebook's code and explanations.